# Chebyshev Polynomial

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
import torch
from IPython.display import Markdown, SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.algorithms.primitives.linear_combination_block_encoding import (
    LinearCombinationBlockEncoding,
)
from flagquantum.algorithms.primitives.unitary_block_encoding import UnitaryBlockEncoding
from flagquantum.compiler._component_trace.resource_model import ComponentCost
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import (
    export_call_graph_dot,
    format_call_graph_markdown,
    render_call_graph_svg,
)
from flagquantum.simulation.unitary import get_unitary

def unitary(gate, alpha=1):
    def emit(ports, workspace, classical_bits, adjoint):
        # Hadamard and X are self-adjoint.
        return (Instruction(gate, ports["q"]),)

    return UnitaryBlockEncoding((("q", 1),), emit, alpha=alpha)

def signature(block):
    widths = dict(system=block.system_bitsize, ancilla=block.ancilla_bitsize,
                  resource=block.resource_bitsize)
    return Signature.build(**{name: width for name, width in widths.items() if width})

def show_calls(root, leaves):
    edges = {(root, name): count for name, count in leaves.items()}
    display(SVG(render_call_graph_svg(export_call_graph_dot(root, edges))))
    display(Markdown(format_call_graph_markdown(root, edges, leaves)))

## Chebyshev block encoding

Given an identity-signal block encoding of $A/\alpha$, the order-$j$ block
encodes $T_j(A/\alpha)$ with normalization one. The first terms are
$T_0(A)=I$, $T_1(A)=A$, $T_2(A)=2A^2-I$ and $T_3(A)=4A^3-3A$.
The circuit alternates a reflection about zero on the ancilla register
with the child and its adjoint. A zero-width ancilla omits the reflections.

[Quantum computing enhanced computational catalysis](https://arxiv.org/abs/2007.14460)
gives the original construction.

In [ ]:
from flagquantum.algorithms.primitives.chebyshev_block_encoding import ChebyshevBlockEncoding

### Example Instances

In [ ]:
average_of_x_and_h = LinearCombinationBlockEncoding(
    (unitary("x"), unitary("h")), (0.5, 0.5), 1
)
even = ChebyshevBlockEncoding(average_of_x_and_h, 4)

In [ ]:
hadamard = unitary("h")
odd = ChebyshevBlockEncoding(hadamard, 5)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Order-four Chebyshev", signature(even), {}),
    ("Order-five Chebyshev", signature(odd), {}),
])))

### Call Graph

In [ ]:
width = even.system_bitsize + even.ancilla_bitsize + even.resource_bitsize
forward = ComponentCost("LCU block", width)
inverse = ComponentCost("LCU block adjoint", width)
model = even.resource_model(forward, inverse)
show_calls("Order-four Chebyshev", {
    call.component.name: call.repetitions for call in model.calls
})

## Scaled Chebyshev block encoding

To encode $T_j(A)$ when the child encodes $A/\alpha$, express $T_j(\alpha x)$ as
a linear combination of $T_k(x)$ with $0\le k\le j$. Convert the original
Chebyshev coefficients to power basis, multiply the $k$-th coefficient by
$\alpha^k$, and convert back. The LCU then uses signed coefficients ordered by
degree, including zero entries. The degree-zero and degree-one cases follow
the source's direct unscaled delegation.

[Explicit Quantum Circuits for Block Encodings of Certain Sparse Matrices](https://arxiv.org/abs/2203.10236)
describes this scaling construction.

In [ ]:
from flagquantum.algorithms.primitives.scaled_chebyshev_block_encoding import (
    ScaledChebyshevBlockEncoding,
)

### Example Instances

In [ ]:
sum_of_x_and_h = LinearCombinationBlockEncoding(
    (unitary("x"), unitary("h")), (1.0, 1.0), 1
)
scaled_even = ScaledChebyshevBlockEncoding(sum_of_x_and_h, 4)
assert scaled_even.alpha == 97

In [ ]:
scaled_odd = ScaledChebyshevBlockEncoding(unitary("h", alpha=3.14), 5)
print("Scaled order-five normalization", scaled_odd.alpha)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Scaled order four", signature(scaled_even), {}),
    ("Scaled order five", signature(scaled_odd), {}),
])))

### Call Graph

In [ ]:
show_calls("Scaled order four", {"Linear combination of Chebyshev blocks": 1})
lcu = scaled_even.linear_combination
assert isinstance(lcu, LinearCombinationBlockEncoding)
print("Coefficients in degree order", lcu.coefficients)

### Verify the odd block through append

The order-five example has no ancilla and applies the Hadamard and its adjoint
in alternating order. Compare the full matrix with Hadamard and then append the
exact inverse.

In [ ]:
def bind_child(ports, work, slots, inverse):
    return hadamard.make_on(
        system=ports["system"], ancilla=ports["ancilla"], resource=ports["resource"],
        workspace=work, classical_bits=slots, adjoint=inverse,
    )

circuit = Circuit(1, dtype=torch.complex128)
circuit.append(odd.make_on(bind_child, system=(0,)))
h = torch.tensor([[1, 1], [1, -1]], dtype=torch.complex128) / 2**0.5
torch.testing.assert_close(get_unitary(circuit), h, atol=2e-12, rtol=0)
circuit.append(odd.make_on(bind_child, system=(0,), adjoint=True))
torch.testing.assert_close(get_unitary(circuit), torch.eye(2, dtype=torch.complex128),
                           atol=2e-12, rtol=0)